<span style="font-family: 'Brush Script MT', cursive; font-size: 28px; color:  #8888CC; font-style: italic;">
    <b>БФБО-01-24 Попов Д.К.</b>
</span>

## **Домашнее задание №14**

### Тема: эмбеддинги, индекс `FAISS`, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

#### **1. Импорты, seed и среда**

In [1]:
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from tqdm.auto import tqdm
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Библиотеки загружены. Готовы к работе с физикой частиц!")

C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Библиотеки загружены. Готовы к работе с физикой частиц!


In [2]:
# Настройки для воспроизводимости
SEED = 161

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Создаем структуру папок
PROJECT_DIRS = ["artifacts"]
for d in PROJECT_DIRS:
    os.makedirs(d, exist_ok=True)

print(f"Структура проекта готова: {PROJECT_DIRS}")

# Выбор устройства для вычислений
try:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

Структура проекта готова: ['artifacts']


In [3]:
# Загрузка FAISS
def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False

FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")
if FAISS_READY:
    import faiss  # type: ignore

SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

In [4]:
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Устройство: {DEVICE}")
print(f"FAISS доступен: {FAISS_READY}")
print(f"Sentence-transformers доступен: {SENTENCE_TRANSFORMERS_READY}")

NumPy: 2.4.2
Pandas: 3.0.1
Устройство: cuda
FAISS доступен: True
Sentence-transformers доступен: True


##### **2. База знаний и первичный анализ**

В качестве базы знаний выбрана подборка из 10 коротких статей по Стандартной модели и физике элементарных частиц.

**Предметная область:** `физика элементарных частиц`.

Данная тема отлично подходит для задачи RAG, так как содержит конкретные факты, точные термины и фундаментальные концепции, по которым удобно проверять качество retrieval и mini-RAG.

In [5]:
documents: List[Dict[str, str]] = [
    {"doc_id": "doc_01","title": "Стандартная модель","text": "Стандартная модель — современная теория, описывающая все известные элементарные частицы и три из четырёх фундаментальных взаимодействий (электромагнитное, слабое и сильное). Она объединяет кварки, лептоны, калибровочные бозоны и бозон Хиггса."},
    {"doc_id": "doc_02","title": "Кварки","text": "Кварки — фундаментальные фермионы, из которых состоят адроны (протоны, нейтроны, мезоны). Существует шесть типов (ароматов) кварков: u, d, c, s, t, b. Они обладают цветовым зарядом и участвуют в сильном взаимодействии."},
    {"doc_id": "doc_03","title": "Лептоны","text": "Лептоны — класс фундаментальных фермионов, не участвующих в сильном взаимодействии. К ним относятся электрон, мюон, тау-лептон и три типа нейтрино. Лептоны делятся на три поколения."},
    {"doc_id": "doc_04", "title": "Фотон","text": "Фотон — безмассовая частица-переносчик электромагнитного взаимодействия. Он является собственным бозоном электромагнитного поля и отвечает за все электромагнитные явления, включая свет."},
    {"doc_id": "doc_05","title": "W- и Z-бозоны","text": "W± и Z-бозоны — массивные частицы-переносчики слабого взаимодействия. W-бозоны отвечают за заряженные слабые процессы, а Z-бозон — за нейтральные. Они были открыты на ускорителе SPS в 1983 году."},
    {"doc_id": "doc_06","title": "Глюоны","text": "Глюоны — безмассовые бозоны, переносчики сильного взаимодействия. Они обладают цветовым зарядом и могут взаимодействовать друг с другом, что приводит к явлению конфайнмента кварков."},
    {"doc_id": "doc_07","title": "Бозон Хиггса","text": "Бозон Хиггса — скалярная частица, отвечающая за механизм спонтанного нарушения электрослабой симметрии. Благодаря ему все массивные частицы Стандартной модели приобретают массу. Открыт на БАК в 2012 году."},
    {"doc_id": "doc_08","title": "Три поколения частиц","text": "Стандартная модель содержит три поколения фермионов. Каждое поколение состоит из двух кварков и двух лептонов. Первое поколение (u, d, e, νe) формирует обычное вещество, два остальных — более тяжёлые аналоги."},
    {"doc_id": "doc_09","title": "Фермионы и бозоны","text": "Все элементарные частицы делятся на фермионы (спин 1/2, подчиняются принципу Паули) и бозоны (целый спин). Фермионы — это кварки и лептоны, бозоны — переносчики взаимодействий и бозон Хиггса."},
    {"doc_id": "doc_10","title": "Квантовая хромодинамика","text": "Квантовая хромодинамика (КХД) — теория сильного взаимодействия в рамках Стандартной модели. Она описывает взаимодействие кварков и глюонов через цветовой заряд и объясняет явление асимптотической свободы."}
]

docs_df = pd.DataFrame(documents)
print("Число документов:", len(docs_df))
display(docs_df[["doc_id", "title"]].head())

Число документов: 10


,doc_id,title
0,doc_01,Стандартная модель
1,doc_02,Кварки
2,doc_03,Лептоны
3,doc_04,Фотон
4,doc_05,W- и Z-бозоны


##### **3. Чанкинг документов**

Текст разбивается на фрагменты (чанки) по количеству слов.

`chunk_size` определяет максимальное количество слов в чанке,

а `overlap` — пересечение слов между соседними чанками, чтобы не разрывать контекст.

In [6]:
def chunk_text(text: str, chunk_size: int = 25, overlap: int = 5) -> List[str]:
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

In [7]:
# Демонстрация чанкинга:
demo_chunks = chunk_text(documents[0]["text"], chunk_size=15, overlap=3)
print(f"Оригинальный текст: {documents[0]['text']}")
for i, c in enumerate(demo_chunks):
    print(f"Чанк {i+1}: {c}")

Оригинальный текст: Стандартная модель — современная теория, описывающая все известные элементарные частицы и три из четырёх фундаментальных взаимодействий (электромагнитное, слабое и сильное). Она объединяет кварки, лептоны, калибровочные бозоны и бозон Хиггса.
Чанк 1: Стандартная модель — современная теория, описывающая все известные элементарные частицы и три из четырёх фундаментальных
Чанк 2: из четырёх фундаментальных взаимодействий (электромагнитное, слабое и сильное). Она объединяет кварки, лептоны, калибровочные бозоны и
Чанк 3: калибровочные бозоны и бозон Хиггса.


##### **4. Эмбеддинги и индекс `FAISS`**

In [8]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

In [9]:
class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(texts, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        return self.fit_documents(texts)

In [10]:
class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

In [11]:
def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device)
        except Exception as e:
            print("Ошибка загрузки SentenceTransformers, используем TF-IDF:", e)
    return TfidfFallbackBackend()

In [12]:
@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object

In [13]:
def build_retriever(docs: List[Dict[str, str]], chunk_size: int = 25, overlap: int = 5, device: str = "cpu") -> RetrieverArtifacts:
    rows = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for i, chunk_txt in enumerate(parts):
            rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": f"{doc['doc_id']}_{i}", "chunk_text": chunk_txt})

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if FAISS_READY:
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
    else:
        index = vectors # Используем саму матрицу для fallback

    return RetrieverArtifacts(backend.backend_name, chunks_df, vectors, backend, index)

In [14]:
def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    q_vec = artifacts.backend.encode_queries([query]).astype(np.float32)
    if FAISS_READY:
        scores, indices = artifacts.index.search(q_vec, top_k)
        scores, indices = scores[0], indices[0]
    else:
        sim = (artifacts.chunk_vectors @ q_vec.T).reshape(-1)
        indices = np.argsort(-sim)[:top_k]
        scores = sim[indices]

    res = []
    for rank, (score, idx) in enumerate(zip(scores, indices), 1):
        row = artifacts.chunks_df.iloc[int(idx)]
        res.append({"rank": rank, "score": float(score), "doc_id": row["doc_id"], "title": row["title"], "chunk_text": row["chunk_text"]})
    return pd.DataFrame(res)

In [15]:
# Построение baseline retriever
base_artifacts = build_retriever(documents, chunk_size=25, overlap=5, device=DEVICE)
print(f"Backend: {base_artifacts.backend_name}")
print(f"Всего чанков: {len(base_artifacts.chunks_df)}")

# Тестовый поиск
display(search_chunks("Что такое бозон Хиггса и для чего он нужен?", base_artifacts, top_k=2))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6207.50it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Всего чанков: 16


,rank,score,doc_id,title,chunk_text
0,1,0.831554,doc_07,Бозон Хиггса,"Бозон Хиггса — скалярная частица, отвечающая з..."
1,2,0.822260,doc_09,Фермионы и бозоны,"лептоны, бозоны — переносчики взаимодействий и..."


##### **5. Контрольные запросы и оценка retrieval**

Подготовлено **8 контрольных запросов** с известными релевантными документами.  
Посчитаны метрики `hit@k`, `recall@k` и `MRR@k` при `top_k=3`.

In [16]:
benchmark_queries = [
    {"query_id": "q01", "query": "Что описывает Стандартная модель элементарных частиц?", "relevant_doc_ids": ["doc_01"]},
    {"query_id": "q02", "query": "Сколько существует типов кварков и какие они?", "relevant_doc_ids": ["doc_02"]},
    {"query_id": "q03", "query": "Какие частицы относятся к лептонам?", "relevant_doc_ids": ["doc_03"]},
    {"query_id": "q04", "query": "Что такое фотон и какое взаимодействие он переносит?", "relevant_doc_ids": ["doc_04"]},
    {"query_id": "q05", "query": "Для чего нужны W- и Z-бозоны?", "relevant_doc_ids": ["doc_05"]},
    {"query_id": "q06", "query": "Что такое глюоны и почему они важны для конфайнмента?", "relevant_doc_ids": ["doc_06"]},
    {"query_id": "q07", "query": "Когда и где был открыт бозон Хиггса?", "relevant_doc_ids": ["doc_07"]},
    {"query_id": "q08", "query": "Сколько поколений фермионов в Стандартной модели?", "relevant_doc_ids": ["doc_08"]},
]

benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,query_id,query,relevant_doc_ids
0,q01,Что описывает Стандартная модель элементарных ...,[doc_01]
1,q02,Сколько существует типов кварков и какие они?,[doc_02]
2,q03,Какие частицы относятся к лептонам?,[doc_03]
3,q04,Что такое фотон и какое взаимодействие он пере...,[doc_04]
4,q05,Для чего нужны W- и Z-бозоны?,[doc_05]
5,q06,Что такое глюоны и почему они важны для конфай...,[doc_06]
6,q07,Когда и где был открыт бозон Хиггса?,[doc_07]
7,q08,Сколько поколений фермионов в Стандартной модели?,[doc_08]


In [17]:
def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered

In [18]:
def evaluate_query(query: str, relevant_doc_ids: List[str], artifacts: RetrieverArtifacts, top_k: int = 3) -> Dict:
    res_df = search_chunks(query, artifacts, top_k=top_k)
    pred_docs = unique_doc_order(res_df)

    hit = int(any(d in pred_docs for d in relevant_doc_ids))
    recall = sum(d in pred_docs for d in relevant_doc_ids) / len(relevant_doc_ids)

    first_rank = next((idx for idx, d in enumerate(pred_docs, 1) if d in relevant_doc_ids), None)
    mrr = 0.0 if first_rank is None else 1.0 / first_rank

    return {"predicted_doc_ids": pred_docs, "hit": hit, "recall": recall, "mrr": mrr, "first_relevant_rank": first_rank}

In [19]:
def evaluate_benchmark(queries: List[Dict], artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    rows = []
    for q in queries:
        eval_res = evaluate_query(q["query"], q["relevant_doc_ids"], artifacts, top_k)
        rows.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "expected_source": ", ".join(q["relevant_doc_ids"]),
            "retrieved_sources": ", ".join(eval_res["predicted_doc_ids"]),
            f"hit_at_k": eval_res["hit"],
            f"recall_at_k": eval_res["recall"],
            f"mrr_at_k": eval_res["mrr"],
            "rank_of_first_relevant": eval_res["first_relevant_rank"]
        })
    return pd.DataFrame(rows)

In [20]:
base_eval_df = evaluate_benchmark(benchmark_queries, base_artifacts, top_k=3)
display(base_eval_df.head(5))

# Сохранение артефакта
base_eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,mrr_at_k,rank_of_first_relevant
0,q01,Что описывает Стандартная модель элементарных ...,doc_01,"doc_01, doc_09, doc_07",1,1.0,1.0,1
1,q02,Сколько существует типов кварков и какие они?,doc_02,"doc_02, doc_08, doc_09",1,1.0,1.0,1
2,q03,Какие частицы относятся к лептонам?,doc_03,"doc_09, doc_03, doc_01",1,1.0,0.5,2
3,q04,Что такое фотон и какое взаимодействие он пере...,doc_04,"doc_04, doc_06, doc_02",1,1.0,1.0,1
4,q05,Для чего нужны W- и Z-бозоны?,doc_05,"doc_05, doc_09, doc_08",1,1.0,1.0,1


##### **6. Небольшой эксперимент с параметрами retrieval**

Сравним `chunk_size = 15` и `chunk_size = 40` при фиксированном `overlap = 5`.

In [21]:
exp_configs = [
    {"chunk_size": 15, "overlap": 5},
    {"chunk_size": 40, "overlap": 5}
]

exp_results = []
for cfg in exp_configs:
    exp_art = build_retriever(documents, chunk_size=cfg["chunk_size"], overlap=cfg["overlap"], device=DEVICE)
    eval_df = evaluate_benchmark(benchmark_queries, exp_art, top_k=3)
    exp_results.append({
        "chunk_size": cfg["chunk_size"],
        "num_chunks": len(exp_art.chunks_df),
        "mean_hit@3": eval_df["hit_at_k"].mean(),
        "mean_MRR@3": eval_df["mrr_at_k"].mean()
    })

display(pd.DataFrame(exp_results))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5999.31it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6038.46it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_size,num_chunks,mean_hit@3,mean_MRR@3
0,15,26,1.0,0.916667
1,40,10,1.0,1.000000


**Вывод по эксперименту:**

Увеличение `chunk_size` с 15 до 40 слов уменьшило количество чанков с 26 до 10, при этом качество retrieval (hit@3 и MRR@3) осталось высоким.  
В небольшом датасете разница не критична, однако на практике слишком мелкий чанкинг может разрывать важные смысловые связи, а слишком крупный — добавлять шум. Оптимальный размер чанка зависит от характера документов и длины типичных запросов.

##### **7. Обновление базы знаний и переиндексация**

Добавим три новых документа, расширяющих базу знаний по Стандартной модели:
- сильное взаимодействие,
- слабое взаимодействие,
- механизм Хиггса и происхождение массы частиц.

Посмотрим, как изменится качество retrieval после переиндексации.

In [22]:
new_documents = [
    {"doc_id": "doc_11","title": "Сильное взаимодействие","text": "Сильное взаимодействие — самое мощное из фундаментальных. Оно удерживает кварки внутри адронов и отвечает за стабильность атомных ядер. Описывается квантовой хромодинамикой."},
    {"doc_id": "doc_12","title": "Слабое взаимодействие","text": "Слабое взаимодействие отвечает за бета-распад, превращение нейтрона в протон и процессы в Солнце. Оно нарушает зеркальную симметрию и является единственным взаимодействием, изменяющим аромат кварков."},
    {"doc_id": "doc_13","title": "Масса частиц и механизм Хиггса","text": "Механизм Хиггса объясняет, почему некоторые частицы имеют массу, а другие (фотон, глюон) — нет. Поле Хиггса заполняет всё пространство и взаимодействует с частицами Стандартной модели."}
]

updated_documents = documents + new_documents
updated_artifacts = build_retriever(updated_documents, chunk_size=25, overlap=5, device=DEVICE)

update_queries = [
    {"query_id": "q11", "query": "Что отвечает за удержание кварков внутри адронов?", "relevant_doc_ids": ["doc_11"]},
    {"query_id": "q12", "query": "Какое взаимодействие нарушает зеркальную симметрию и отвечает за бета-распад?", "relevant_doc_ids": ["doc_12"]},
    {"query_id": "q13", "query": "Как механизм Хиггса объясняет наличие массы у частиц?", "relevant_doc_ids": ["doc_13"]},
]

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6162.27it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# Сравнение retrieval до и после обновления базы
eval_before = evaluate_benchmark(update_queries, base_artifacts, top_k=3)
eval_after = evaluate_benchmark(update_queries, updated_artifacts, top_k=3)

compare_df = pd.merge(
    eval_before[["query", "expected_source", "retrieved_sources"]].rename(columns={"retrieved_sources": "before_retrieved_sources"}),
    eval_after[["query", "retrieved_sources"]].rename(columns={"retrieved_sources": "after_retrieved_sources"}),
    on="query"
)
compare_df["changed"] = compare_df["before_retrieved_sources"] != compare_df["after_retrieved_sources"]

display(compare_df)

# Сохранение артефакта
compare_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

,query,expected_source,before_retrieved_sources,after_retrieved_sources,changed
0,Что отвечает за удержание кварков внутри адронов?,doc_11,"doc_09, doc_02, doc_06","doc_09, doc_02, doc_06",False
1,Какое взаимодействие нарушает зеркальную симме...,doc_12,"doc_08, doc_05, doc_09","doc_12, doc_08, doc_05",True
2,Как механизм Хиггса объясняет наличие массы у ...,doc_13,"doc_07, doc_09, doc_01","doc_13, doc_07, doc_09",True


##### **8. Mini-RAG**

Надо построить pipeline, который извлекает текст, формирует контекст и с помощью механизма извлечения `(extractive summaries)` отвечает на вопрос.

*Примечание: Чтобы не усложнять зависимости LLM, генератор формирует ответ из наиболее релевантных предложений из найденного контекста.*

In [24]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

In [25]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []
    for _, row in retrieved.iterrows():
        block = f"[Doc: {row['doc_id']}] {row['chunk_text']}"
        context_blocks.append(block)
    return "\n".join(context_blocks), retrieved

In [26]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    # Удаляем служебные теги для чистого текста
    content_lines = [re.sub(r"\[Doc: .*?\]\s*", "", line) for line in raw_lines]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 3]
    if not sentence_pool:
        return "Недостаточно контекста для ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]
    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used = set()

    for idx in ranked_idx:
        if scores[idx] <= 0: continue
        sent = sentence_pool[idx]
        norm_sent = sent.lower()
        if norm_sent not in used:
            used.add(norm_sent)
            selected_sentences.append(sent)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В контексте нет ответа."
    return " ".join(selected_sentences)

In [27]:
def mini_rag_answer(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> dict:
    context, retrieved = build_context_from_retrieval(query, artifacts, top_k)
    answer = generate_answer_from_context(query, context)
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(retrieved["doc_id"].unique())
    }

In [28]:
# Демонстрация Mini-RAG:
rag_queries = [
    "Что такое бозон Хиггса и зачем он нужен?",
    "Чем отличаются фермионы от бозонов?",
    "Почему кварки не наблюдаются в свободном состоянии?",
    "Какие частицы переносят электромагнитное взаимодействие?",
    "Что такое три поколения частиц в Стандартной модели?"
]

rag_results = []
for q in rag_queries:
    res = mini_rag_answer(q, updated_artifacts)
    rag_results.append(res)
    print(f"Q: {res['question']}\nA: {res['answer']}\nSources: {res['retrieved_sources']}\n{'-'*50}")

rag_df = pd.DataFrame(rag_results)

# Сохранение артефакта
rag_df.to_csv("artifacts/rag_examples.csv", index=False)

Q: Что такое бозон Хиггса и зачем он нужен?
A: лептоны, бозоны — переносчики взаимодействий и бозон Хиггса. Бозон Хиггса — скалярная частица, отвечающая за механизм спонтанного нарушения электрослабой симметрии.
Sources: doc_07, doc_09, doc_13
--------------------------------------------------
Q: Чем отличаются фермионы от бозонов?
A: Фермионы — это кварки и лептоны, бозоны — переносчики взаимодействий Все элементарные частицы делятся на фермионы (спин 1/2, подчиняются принципу Паули) и бозоны (целый спин).
Sources: doc_09, doc_01
--------------------------------------------------
Q: Почему кварки не наблюдаются в свободном состоянии?
A: Фермионы — это кварки и лептоны, бозоны — переносчики взаимодействий
Sources: doc_09, doc_12, doc_06
--------------------------------------------------
Q: Какие частицы переносят электромагнитное взаимодействие?
A: Сильное взаимодействие — самое мощное из фундаментальных. Все элементарные частицы делятся на фермионы (спин 1/2, подчиняются принципу Паул

##### **9. Краткий анализ ошибок**

### Успешные кейсы
Mini-RAG хорошо справляется с вопросами, где ключевые термины напрямую присутствуют в базе знаний:
- «Что такое бозон Хиггса и зачем он нужен?»
- «Сколько поколений фермионов в Стандартной модели?»
- «Что такое фотон и какое взаимодействие он переносит?»

В этих случаях retriever уверенно находит правильные документы, а extractive-генератор собирает релевантные предложения.

### Проблемные и пограничные кейсы

1. **Запрос: «Почему кварки не наблюдаются в свободном состоянии?»**  
   Ответ получился слишком общим («Фермионы — это кварки и лептоны…»).  
   Система нашла документы про фермионы и кварки, но не смогла явно извлечь понятие **конфайнмента** и роль сильного взаимодействия. Причина — чанк не содержал достаточного контекста.

2. **Запрос: «Какие частицы переносят электромагнитное взаимодействие?»**  
   Retriever вернул информацию об ином типе взаимодействия, перепутав электромагнитное с сильным. 
   Extractive-подход иногда «приклеивает» соседние предложения с высокой TF-IDF-похожестью и/или выбирает неправильный токен с похожим смыслом.

### Основные причины ошибок

- **Ограничения чанкинга**: при `chunk_size=25` некоторые важные понятия (например, конфайнмент) могут оказаться на границе чанков и терять контекст.
- **Слабый генератор**: текущий extractive-метод на основе TF-IDF хорошо находит ключевые слова, но плохо связывает их в логичный ответ. Настоящая LLM справилась бы с этим гораздо лучше.
- **Отсутствие reranking**: иногда релевантный чанк оказывается не на первом месте, и в контекст попадает шум.
- **Небольшой объём базы**: в обучающем датасете не хватает глубины объяснений, поэтому на сложные концептуальные вопросы ответы получаются поверхностными.

### Вывод по качеству mini-RAG
Retrieval-слой работает достаточно надёжно (hit@3 ≈ 1.0 на большинстве запросов).  
Главное узкое место — этап генерации ответа. Переход от простого извлечения предложений к полноценной генеративной модели (например, через промпт с контекстом) сильно улучшит качество финальных ответов.